# DPL Tag Classifier — Fine-tuning Microsoft DeBERTa-v3-base

**Model:** `microsoft/deberta-v3-base`  
**Task:** Multi-class text classification (76 DPL tags)  
**Dataset:** Synthetic DPL training data generated by `generate_dpl_data.py`  
**Hardware:** NVIDIA GPU required

## Notebook Structure
1. Environment & GPU check  
2. Install dependencies  
3. Load & inspect dataset  
4. Label encoding  
5. Tokenisation & HuggingFace Dataset  
6. Model setup  
7. Training  
8. Evaluation — classification report & confusion matrix  
9. Save model  
10. Inference helper  

## 1. Environment & GPU Check

In [ ]:
import subprocess, sys

# Quick sanity check before installing anything
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout[:800])
else:
    print("WARNING: nvidia-smi not found — make sure CUDA drivers are installed.")

## 2. Install Dependencies

> Run this cell once. Restart the kernel afterwards if packages were freshly installed.

In [ ]:
%pip install -q \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 \
    transformers>=4.40.0 \
    datasets>=2.19.0 \
    accelerate>=0.29.0 \
    sentencepiece \
    protobuf \
    scikit-learn \
    pandas \
    seaborn \
    matplotlib

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Active device   : {DEVICE}")

## 3. Load & Inspect Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATASETS_DIR = "datasets"

train_df = pd.read_csv(f"{DATASETS_DIR}/dpl_train.csv")
val_df   = pd.read_csv(f"{DATASETS_DIR}/dpl_val.csv")
test_df  = pd.read_csv(f"{DATASETS_DIR}/dpl_test.csv")

print(f"Train : {len(train_df):,} rows")
print(f"Val   : {len(val_df):,} rows")
print(f"Test  : {len(test_df):,} rows")
print(f"\nUnique DPL tags: {train_df['dpl_tag'].nunique()}")
train_df.head(10)

In [ ]:
# Distribution plot — every tag should be roughly balanced
fig, ax = plt.subplots(figsize=(18, 4))
train_df["dpl_tag"].value_counts().sort_index().plot(kind="bar", ax=ax, width=0.8)
ax.set_title("Training samples per DPL tag")
ax.set_xlabel("DPL Tag")
ax.set_ylabel("Count")
ax.tick_params(axis="x", labelsize=7, rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Description length distribution
train_df["desc_len"] = train_df["description"].str.len()
print(train_df["desc_len"].describe())
train_df["desc_len"].hist(bins=40, figsize=(10, 3))
plt.title("Description character length distribution")
plt.xlabel("Characters")
plt.show()

## 4. Label Encoding

In [ ]:
import json
import os
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Fit on the full sorted tag list so indices are stable regardless of split contents
all_tags = sorted(train_df["dpl_tag"].unique().tolist())
le.fit(all_tags)

NUM_LABELS = len(le.classes_)
print(f"Number of classes: {NUM_LABELS}")
print("First 10 classes:", le.classes_[:10].tolist())

# Encode labels
train_df["label"] = le.transform(train_df["dpl_tag"])
val_df["label"]   = le.transform(val_df["dpl_tag"])
test_df["label"]  = le.transform(test_df["dpl_tag"])

# id2label / label2id mappings for the model config
id2label = {int(i): str(c) for i, c in enumerate(le.classes_)}
label2id = {str(c): int(i) for i, c in enumerate(le.classes_)}

# Persist the label map so we can decode predictions outside this notebook
os.makedirs("models/deberta", exist_ok=True)
with open("models/deberta/label_classes.json", "w") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, indent=2)
print("Label map saved to models/deberta/label_classes.json")

## 5. Tokenisation & HuggingFace Dataset

In [ ]:
from transformers import AutoTokenizer
import importlib

# DeBERTa-v3 requires protobuf to load its SentencePiece tokenizer correctly.
# Without it, HuggingFace falls back to a different tokenizer that produces
# wrong token IDs, causing the model to learn nothing and produce near-random predictions.
if importlib.util.find_spec("google.protobuf") is None:
    raise EnvironmentError(
        "protobuf is not installed.\n"
        "Run:  pip install protobuf\n"
        "Then restart the kernel before continuing."
    )
print("protobuf OK")

MODEL_CHECKPOINT = "microsoft/deberta-v3-base"
MAX_LENGTH = 128   # DPL descriptions are short; 128 tokens is more than enough

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
assert "DebertaV2Tokenizer" in tokenizer.__class__.__name__, (
    f"Wrong tokenizer loaded: {tokenizer.__class__.__name__}. "
    "Expected DebertaV2Tokenizer — protobuf may not have been picked up. "
    "Restart the kernel and try again."
)
print(f"Tokenizer loaded: {tokenizer.__class__.__name__} ✓")

In [ ]:
from datasets import Dataset, DatasetDict

def df_to_hf_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_dict({
        "text":  df["description"].tolist(),
        "label": df["label"].tolist(),
    })

raw_datasets = DatasetDict({
    "train": df_to_hf_dataset(train_df),
    "val":   df_to_hf_dataset(val_df),
    "test":  df_to_hf_dataset(test_df),
})
print(raw_datasets)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

tokenized = raw_datasets.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
)
tokenized.set_format("torch")
print(tokenized)

## 6. Model Setup

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

## 7. Training

### Hyperparameters

| Parameter | Value | Notes |
|---|---|---|
| Learning rate | 2e-5 | Standard for DeBERTa fine-tuning |
| Batch size | 32 | Reduce to 16 if OOM |
| Epochs | 5 | Increase if val loss still falling |
| Warmup ratio | 0.1 | 10% warmup steps |
| Weight decay | 0.01 | Light regularisation |
| FP16 | True | Requires NVIDIA GPU |

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# ---- Compute metrics callback ----
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_score(labels, preds)
    f1    = f1_score(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "f1_weighted": f1}

# ---- Warmup steps (replaces deprecated warmup_ratio) ----
EPOCHS          = 5
BATCH_SIZE      = 32
n_train         = len(tokenized["train"])
steps_per_epoch = max(1, n_train // BATCH_SIZE)
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = int(total_steps * 0.1)
print(f"Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")

# ---- Training arguments ----
training_args = TrainingArguments(
    output_dir                  = "models/deberta/checkpoints",
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_steps                = warmup_steps,
    lr_scheduler_type           = "cosine",
    bf16                        = torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16                        = torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_weighted",
    greater_is_better           = True,
    logging_steps               = 50,
    report_to                   = "none",
    save_total_limit            = 2,
    seed                        = 42,
    dataloader_num_workers      = 4,     # 4 on Linux/Ubuntu; change to 0 if running on Windows
)

# ---- Trainer (tokenizer -> processing_class in transformers v5) ----
trainer = Trainer(
    model              = model,
    args               = training_args,
    train_dataset      = tokenized["train"],
    eval_dataset       = tokenized["val"],
    processing_class   = tokenizer,
    compute_metrics    = compute_metrics,
)

print("Trainer ready. Starting training...")

In [ ]:
train_result = trainer.train()
print(train_result)

In [ ]:
# Plot training loss curve from log history
log_history = trainer.state.log_history

train_losses = [(e["epoch"], e["loss"]) for e in log_history if "loss" in e]
val_losses   = [(e["epoch"], e["eval_loss"]) for e in log_history if "eval_loss" in e]
val_f1s      = [(e["epoch"], e["eval_f1_weighted"]) for e in log_history if "eval_f1_weighted" in e]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

if train_losses:
    ax1.plot(*zip(*train_losses), label="Train loss", alpha=0.7)
if val_losses:
    ax1.plot(*zip(*val_losses), label="Val loss", marker="o")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

if val_f1s:
    ax2.plot(*zip(*val_f1s), label="Val F1 (weighted)", marker="o", color="green")
ax2.set_title("Validation F1 (Weighted)")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.show()

## 8. Evaluation on Test Set

In [ ]:
from sklearn.metrics import classification_report
import json

# Run inference on the test set
test_output   = trainer.predict(tokenized["test"])
test_preds    = np.argmax(test_output.predictions, axis=-1)
test_labels   = test_output.label_ids

# Convert numeric ids back to DPL tag names
pred_tags  = le.inverse_transform(test_preds)
true_tags  = le.inverse_transform(test_labels)
tag_names  = le.classes_.tolist()

report = classification_report(true_tags, pred_tags, target_names=tag_names, zero_division=0)
print(report)

# Save report
with open("models/deberta/classification_report_test.txt", "w") as f:
    f.write(report)
print("Saved classification report.")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Build per-class accuracy bar chart (easier to read than a 76×76 matrix)
from sklearn.metrics import classification_report as cr_dict

report_dict = cr_dict(true_tags, pred_tags, output_dict=True, zero_division=0)

# Per-class F1
per_class_f1 = {
    tag: report_dict[tag]["f1-score"]
    for tag in tag_names
    if tag in report_dict
}

f1_series = pd.Series(per_class_f1).sort_index()

fig, ax = plt.subplots(figsize=(20, 5))
colors = ["#d62728" if v < 0.80 else "#2ca02c" for v in f1_series.values]
f1_series.plot(kind="bar", ax=ax, color=colors, width=0.8)
ax.axhline(0.80, color="orange", linestyle="--", linewidth=1.2, label="0.80 threshold")
ax.set_title("Per-class F1 Score on Test Set (red = below 0.80)")
ax.set_xlabel("DPL Tag")
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", labelsize=7, rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

# Tags below 0.80 F1 — targets for more training data
weak_tags = f1_series[f1_series < 0.80].sort_values()
print(f"\nTags below F1=0.80 ({len(weak_tags)}):")
print(weak_tags.to_string())

In [ ]:
# Confusion matrix for the most confused tag pairs
# Full 76×76 is hard to read — show only the top 20 most-confused classes

cm = confusion_matrix(true_tags, pred_tags, labels=tag_names)
cm_df = pd.DataFrame(cm, index=tag_names, columns=tag_names)

# Find most confused pairs (off-diagonal, symmetric)
np.fill_diagonal(cm, 0)
confusion_pairs = []
for i, ti in enumerate(tag_names):
    for j, tj in enumerate(tag_names):
        if i < j and (cm[i, j] + cm[j, i]) > 0:
            confusion_pairs.append((ti, tj, cm[i, j] + cm[j, i]))

top_confused = sorted(confusion_pairs, key=lambda x: -x[2])[:20]
print("Top 20 confused tag pairs:")
for a, b, n in top_confused:
    print(f"  {a} ↔ {b}  ({n} errors)")

In [ ]:
# Heatmap of the top confused tags only
if top_confused:
    confused_tags = sorted(set(
        t for pair in top_confused for t in pair[:2]
    ))
    sub_cm = cm_df.loc[confused_tags, confused_tags]

    fig, ax = plt.subplots(figsize=(max(8, len(confused_tags)), max(6, len(confused_tags) * 0.7)))
    sns.heatmap(
        sub_cm, annot=True, fmt="d", cmap="YlOrRd",
        linewidths=0.5, ax=ax
    )
    ax.set_title("Confusion matrix — most confused DPL tag pairs")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.show()

In [ ]:
# Save per-class metrics to JSON
metrics_output = {
    "overall": {
        "accuracy": report_dict["accuracy"],
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
    },
    "per_class": {
        tag: {
            "precision": report_dict[tag]["precision"],
            "recall":    report_dict[tag]["recall"],
            "f1":        report_dict[tag]["f1-score"],
            "support":   report_dict[tag]["support"],
        }
        for tag in tag_names if tag in report_dict
    },
}

with open("models/deberta/metrics.json", "w") as f:
    json.dump(metrics_output, f, indent=2)

print("=" * 50)
print(f"  Overall accuracy  : {metrics_output['overall']['accuracy']:.4f}")
print(f"  Macro F1          : {metrics_output['overall']['macro_f1']:.4f}")
print(f"  Weighted F1       : {metrics_output['overall']['weighted_f1']:.4f}")
print("=" * 50)

## 9. Save Model

In [ ]:
MODEL_SAVE_PATH = "models/deberta/model"

trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print(f"Model saved to: {MODEL_SAVE_PATH}")
print("Contents:")
for f in os.listdir(MODEL_SAVE_PATH):
    size_kb = os.path.getsize(os.path.join(MODEL_SAVE_PATH, f)) / 1024
    print(f"  {f:<40} {size_kb:>8.1f} KB")

## 10. Inference Helper

Load the saved model and classify new descriptions.

In [ ]:
from transformers import pipeline
import json

MODEL_SAVE_PATH = "models/deberta/model"   # re-run this cell independently if needed

# Load id2label map
with open("models/deberta/label_classes.json") as f:
    label_map = json.load(f)

# HuggingFace pipeline — easiest inference interface
classifier = pipeline(
    task="text-classification",
    model=MODEL_SAVE_PATH,
    tokenizer=MODEL_SAVE_PATH,
    device=0 if torch.cuda.is_available() else -1,
    top_k=3,   # return top-3 predictions with scores
)

print("Classifier loaded.")

In [ ]:
# ---- Test with sample descriptions ----
test_descriptions = [
    "INV-55234 – Deloitte audit services FY2025",
    "Monthly payroll – March 2026 – Finance",
    "Interest charged on HSBC overdraft – January",
    "Google Ads campaign – Q2 2026",
    "Office rent – London HQ – April 2026",
    "Redundancy costs – IT department – 2025",
    "KPMG tax advisory fees",
    "AWS cloud hosting – February subscription",
    "Staff Christmas party expense – 2025",
    "FX loss on USD settlement – AP-78341",
]

results = classifier(test_descriptions)

print(f"{'Description':<55}  {'Top Prediction':<10}  {'Score':>6}")
print("-" * 80)
for desc, preds in zip(test_descriptions, results):
    top = preds[0]
    print(f"{desc:<55}  {top['label']:<10}  {top['score']:>6.3f}")

In [ ]:
# ---- Detailed top-3 for a single description ----
desc = "Interest income received from Barclays savings account"
preds = classifier(desc)[0]

print(f"Description: {desc}\n")
print(f"{'Rank':<6} {'DPL Tag':<10} {'Score':>7}")
print("-" * 28)
for rank, p in enumerate(preds, 1):
    print(f"{rank:<6} {p['label']:<10} {p['score']:>7.4f}")

---

## Summary

| Step | Detail |
|---|---|
| Model | `microsoft/deberta-v3-base` |
| Classes | 76 DPL tags (DPL001–DPL076) |
| Max token length | 128 |
| Optimiser | AdamW (HF default) |
| Scheduler | Cosine with 10% warmup |
| Mixed precision | FP16 (GPU) |
| Best checkpoint | Loaded by `load_best_model_at_end=True` |
| Saved to | `models/deberta/model/` |

### Next steps

- If any tag has F1 < 0.80, add more templates for that tag in `generate_dpl_data.py` and regenerate
- Increase `--n` to 500–1000 samples per tag for stronger generalisation
- Try `microsoft/deberta-v3-large` for higher accuracy at the cost of longer training
- Add real ERP descriptions to the training set to close the synthetic-to-real domain gap